# Manual section / ROI annotation — Napari

Interactive tool to manually draw regions of interest (ROIs) on a spatial
section using [Napari](https://napari.org/), then split the section into
separate `.h5ad` files per ROI.

Use this when automated clustering (BANKSY, Leiden) doesn't cleanly separate
regions you can recognise by eye — for example splitting a single slide that
contains multiple physically distinct tissue sections, or manually delimiting
a lesion/scar boundary.

Two annotation modes are provided:
1. **Without gene expression** — draw polygons directly on the spatial point cloud.
2. **With gene expression** — highlight candidate cells based on a marker gene
   signature (e.g. fibrotic or immune markers) before drawing polygons, which
   makes it easier to trace a region defined by expression rather than shape alone.

**Note:** this notebook requires a graphical environment (Napari opens a desktop
window) — it will not run headless or in a remote/server-only Jupyter session.

In [ ]:
import os, re
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings("ignore") 

import scipy.sparse as sparse
from scipy.io import mmread
from scipy.stats import pearsonr, pointbiserialr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

import seaborn as sns
import scanpy as sc
sc.logging.print_header()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
plt.rcParams["font.family"] = "Arial"
sns.set_style("white")

import random
# Note that BANKSY itself is deterministic, here the seeds affect the umap clusters and leiden partition
seed = 1234
np.random.seed(seed)
random.seed(seed)

import scipy.sparse as sp

### Load data

In [ ]:
adata_all = sc.read_h5ad("/path/to/your/merged_dataset.h5ad")

In [ ]:
adata.layers
adata.uns

In [ ]:
# Choose the slide/section to annotate
SLIDE_ID = "your_slide_id"  # must match a value in adata.obs["sample"]

adata = adata_all[adata_all.obs['sample'] == SLIDE_ID].copy()

if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)

x = adata.obs["x"]
y = adata.obs["y"]

adata.obsm["spatial"] = np.vstack([adata.obs["x"], adata.obs["y"]]).T

spatial = ('x', 'y', 'coord_xy')

 ### Manual section delimitation - Use the polygon tool to draw around points, Press 'q' to save clusters and close Napari. The ROI are then directly saved

### Manual annotation without gene expression

In [ ]:
import numpy as np
import napari
from shapely.geometry import Point, Polygon

def manual_cluster_napari(adata, slide_to_use, spatial_key='spatial', slide_col='sample', cluster_col_prefix='manual_cluster_slide'):
    # Filter for chosen slide
    idx = adata.obs[slide_col] == slide_to_use
    coords = adata.obsm[spatial_key][idx]

    # Prepare column for cluster labels
    col_name = f"{cluster_col_prefix}_{slide_to_use}"
    if col_name not in adata.obs.columns:
        adata.obs[col_name] = -1

    # Create Napari viewer
    viewer = napari.Viewer()
    viewer.add_points(
        coords,
        size=2,
        face_color='white',
        name='cells'
    )
    viewer.add_shapes(name='clusters')  # Explicit name

    # Store metadata so we can link back to adata later
    viewer.layers['cells'].metadata['adata_idx'] = np.where(idx)[0]
    viewer.layers['cells'].metadata['col_name'] = col_name
    viewer.layers['cells'].metadata['adata'] = adata

    print("\n--- Napari Instructions ---")
    print("1. Select the 'clusters' layer.")
    print("2. Use the polygon tool to draw around points.")
    print("3. Press 'q' to save clusters and close Napari.\n")

    @viewer.bind_key('q')  # Press 'q' to quit and save clusters
    def save_clusters_and_quit(v):
        shapes = v.layers['clusters'].data
        if len(shapes) == 0:
            print("No polygons drawn — nothing to save.")
            v.close()
            return

        adata_idx = v.layers['cells'].metadata['adata_idx']
        col_name = v.layers['cells'].metadata['col_name']
        adata_ref = v.layers['cells'].metadata['adata']

        for cluster_id, polygon_coords in enumerate(shapes):
            poly = Polygon(polygon_coords)
            for local_idx, point in enumerate(v.layers['cells'].data):
                if poly.contains(Point(point)):
                    global_idx = adata_idx[local_idx]
                    adata_ref.obs.at[adata_ref.obs.index[global_idx], col_name] = cluster_id

        print(f"Clusters saved to adata.obs['{col_name}']")
        v.close()

    return viewer


In [ ]:
viewer = manual_cluster_napari(adata, slide_to_use=SLIDE_ID)
napari.run()

### Manual annotation with gene expression

In [ ]:
import numpy as np
import napari
from shapely.geometry import Point, Polygon


def napari_subset_by_gene_signature(
    adata,
    slide_to_use,
    genes,
    spatial_key="spatial",
    slide_col="sample",
    expr_layer=None,
    expr_mode="sum",
    threshold_percentile=90,
    out_col_prefix="manual_area",
    point_size_all=1,
    point_size_candidates=0.1,
    log10_transform=True,
    eps=1e-6,
):
    # ---------------- slide filter ----------------
    idx = (adata.obs[slide_col] == slide_to_use).to_numpy()
    if idx.sum() == 0:
        raise ValueError(f"No cells for slide '{slide_to_use}' in adata.obs['{slide_col}'].")

    coords = np.asarray(adata.obsm[spatial_key][idx], dtype=float)
    coords = coords[:, :2]  # (N,2)

    global_indices_for_slide = np.where(idx)[0]

    # ---------------- signature ----------------
    def _get_gene_vec(gene):
        if gene not in adata.var_names:
            raise ValueError(f"Gene '{gene}' not found")
        X = adata.layers[expr_layer] if expr_layer is not None else adata.X
        gix = np.where(adata.var_names == gene)[0][0]
        v = X[idx, gix]
        if hasattr(v, "toarray"):
            v = v.toarray().ravel()
        else:
            v = np.asarray(v).ravel()
        return v.astype(float)

    genes = tuple(genes)
    stack = np.vstack([_get_gene_vec(g) for g in genes])
    sig_raw = stack.sum(axis=0) if expr_mode == "sum" else stack.mean(axis=0)
    sig = np.log10(sig_raw + eps) if log10_transform else sig_raw.astype(float)

    sig_name = f"sig_{'_'.join(genes)}_{expr_mode}_{slide_to_use}"
    if sig_name not in adata.obs.columns:
        adata.obs[sig_name] = np.nan
    adata.obs.loc[adata.obs.index[idx], sig_name] = sig

    print(f"\n[signature] {sig_name}")
    print("  min:", float(np.nanmin(sig)))
    print("  max:", float(np.nanmax(sig)))
    print("  mean:", float(np.nanmean(sig)))

    # ---------------- output column ----------------
    # After saving:
    #   0 = outside any ROI
    #   1 = ROI #1 (first polygon)
    #   2 = ROI #2 (second polygon)
    #   ...
    out_col = f"{out_col_prefix}_slide_{slide_to_use}"
    if out_col not in adata.obs.columns:
        adata.obs[out_col] = np.nan
    adata.obs.loc[adata.obs.index[idx], out_col] = 0

    # ---------------- napari ----------------
    viewer = napari.Viewer()

    all_layer = viewer.add_points(
        coords,
        size=float(point_size_all),
        face_color="white",
        symbol="disc",
        name="all_cells",
    )

    candidates_layer = viewer.add_points(
        np.empty((0, 2), dtype=float),
        size=float(point_size_candidates),
        symbol="disc",
        name="candidates",
    )

    polygons_layer = viewer.add_shapes(name="polygons")

    # Force camera to points so you draw on top of them
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    viewer.camera.center = (mins + maxs) / 2
    viewer.reset_view()

    def _apply_candidate_threshold(pct):
        thr = float(np.nanpercentile(sig, pct))
        mask = sig >= thr
        candidates_layer.data = coords[mask]
        candidates_layer.properties = {"sig": sig[mask]}
        candidates_layer.face_color = "sig"
        try:
            candidates_layer.face_color_mode = "colormap"
        except Exception:
            pass
        try:
            candidates_layer.face_colormap = "Reds"
        except Exception:
            pass
        print(f"[candidates] percentile={pct:.1f}, threshold={thr:.4f}, n={int(mask.sum())}")

    _apply_candidate_threshold(float(threshold_percentile))

    # ---------------- ROI test helpers ----------------
    def _poly_to_shapely(poly_2d, assume="yx"):
        """
        poly_2d is a napari shape (N,2) in either (y,x) or (x,y).
        Return shapely Polygon in (x,y).
        """
        poly_2d = np.asarray(poly_2d, float)
        poly_2d = poly_2d[:, :2]
        if assume == "yx":
            poly_xy = np.column_stack((poly_2d[:, 1], poly_2d[:, 0]))
        else:  # assume == "xy"
            poly_xy = np.column_stack((poly_2d[:, 0], poly_2d[:, 1]))
        return Polygon(poly_xy)

    def _points_to_xy(points_2d, assume="yx"):
        points_2d = np.asarray(points_2d, float)[:, :2]
        if assume == "yx":
            return np.column_stack((points_2d[:, 1], points_2d[:, 0]))
        else:
            return np.column_stack((points_2d[:, 0], points_2d[:, 1]))

    def _best_axis_assumption(points_2d, polygons_list):
        """
        Decide whether data are (y,x) or (x,y) by checking which assumption yields more hits.
        """
        pts = np.asarray(points_2d, float)[:, :2]
        polys = [np.asarray(p, float)[:, :2] for p in polygons_list if len(p) >= 3]

        def count_hits(assume):
            pts_xy = _points_to_xy(pts, assume)
            hit_any = np.zeros(pts_xy.shape[0], dtype=bool)
            for poly in polys:
                P = _poly_to_shapely(poly, assume)
                for i, (x, y) in enumerate(pts_xy):
                    if not hit_any[i] and P.covers(Point(float(x), float(y))):
                        hit_any[i] = True
            return int(hit_any.sum())

        n_yx = count_hits("yx")
        n_xy = count_hits("xy")
        chosen = "xy" if n_xy > n_yx else "yx"
        print(f"[roi-debug] hits if assume yx={n_yx}, assume xy={n_xy} -> using '{chosen}'")
        return chosen

    # ---------------- Save ROI IDs (1..n_polygons) ----------------
    from magicgui import magicgui

    @magicgui(call_button="Save ROI IDs (1..n)")
    def save_roi_ids_button():
        polys = polygons_layer.data
        if len(polys) == 0:
            print("[save] No polygons drawn.")
            return

        pts = np.asarray(all_layer.data, float)[:, :2]

        # pick best axis assumption once
        assume = _best_axis_assumption(pts, polys)

        # Convert all points to shapely (x,y) once
        pts_xy = _points_to_xy(pts, assume)

        # Label array for slide cells: 0 outside, 1..K for each polygon
        labels = np.zeros(pts.shape[0], dtype=int)

        # For each polygon in draw order, assign roi_id = 1..K
        # If polygons overlap, first polygon wins (you can change if you prefer last-wins)
        for roi_id, poly in enumerate(polys, start=1):
            poly = np.asarray(poly, float)
            if poly.ndim != 2 or poly.shape[1] < 2 or len(poly) < 3:
                continue
            P = _poly_to_shapely(poly[:, :2], assume)

            for i, (x, y) in enumerate(pts_xy):
                if labels[i] == 0 and P.covers(Point(float(x), float(y))):
                    labels[i] = roi_id

        # Save back to adata (only for this slide)
        adata.obs.loc[adata.obs.index[idx], out_col] = labels

        # Print counts (ignore outside if you don't care)
        roi_ids = [i for i in np.unique(labels) if i != 0]
        counts = {i: int((labels == i).sum()) for i in roi_ids}
        print(f"[save] Saved {len(roi_ids)} ROIs to adata.obs['{out_col}'] (IDs 1..{len(roi_ids)})")
        print("       cells per ROI:", counts)

    viewer.window.add_dock_widget(save_roi_ids_button, area="right")

    print("\n--- Workflow ---")
    print("1) Draw multiple polygons in the 'polygons' layer ON TOP of the points")
    print("2) Click 'Save ROI IDs (1..n)'\n")
    print(f"Output column: adata.obs['{out_col}']")
    print("  0 = outside any ROI")
    print("  1..n = ROI id in draw order")

    return viewer


In [ ]:
# Example gene signature — replace with markers relevant to your ROI of interest
viewer = napari_subset_by_gene_signature(
    adata,
    slide_to_use=SLIDE_ID,
    genes=("Gene1", "Gene2", "Gene3"),   # ← replace with your marker genes
    expr_layer=None,               # log1p in X
    expr_mode="sum",
    threshold_percentile=90,
    point_size_all=1,
    point_size_candidates=20
)

## Cluster processing

In [ ]:
# keep only cells inside ROI
roi_col = f"manual_area_slide_{SLIDE_ID}"  # matches the out_col printed by the napari tool
adata_roi = adata[adata.obs[roi_col].astype(int) > 0].copy()

print("Original:", adata.n_obs, "cells")
print("ROI only:", adata_roi.n_obs, "cells")

roi_ids = sorted([i for i in adata.obs[roi_col].dropna().unique() if i > 0])
roi_name_map = {i: f"ROI_{i}" for i in roi_ids}

adata.obs["roi_name"] = adata.obs[roi_col].map(roi_name_map)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

spatial_key = "spatial"
coords = np.asarray(adata.obsm[spatial_key])[:, :2]

mask = adata.obs["roi_name"].notna()
coords_roi = coords[mask]
labels_roi = adata.obs.loc[mask, "roi_name"].values

unique_labels = np.unique(labels_roi)

plt.figure(figsize=(6, 6))

for lab in unique_labels:
    pts = coords_roi[labels_roi == lab]
    plt.scatter(pts[:, 1], pts[:, 0], s=1, label=lab)

plt.gca().invert_yaxis()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Napari-drawn ROIs")
plt.legend(markerscale=5, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.show()


In [ ]:
# Rename ROI IDs with meaningful labels.
# Edit this mapping to match your experimental design 
# (e.g. injury type, timepoint, replicate number).
manual_anno = {
    1: "condition_day_rep1",
    2: "condition_day_rep2",
    3: "condition_day_rep3",
    4: "condition_day_rep4",
}

adata_roi.obs['manual_anno'] = adata_roi.obs[roi_col].map(manual_anno)

In [ ]:
# Visualize the manual_anno on the sections
col_name = "manual_anno"
idx = adata_roi.obs['sample'] == SLIDE_ID
coords = adata_roi.obsm['spatial'][idx]
clusters = adata_roi.obs.loc[idx, col_name]

unique_labels = clusters.unique()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], s=10, c=clusters.astype('category').cat.codes, cmap='tab10')
plt.gca().invert_yaxis()
plt.title(f"Manual Anno Clusters for {SLIDE_ID}")
plt.xlabel('X')
plt.ylabel('Y')

# Annotate cluster names at centroids
for label in unique_labels:
    mask = clusters == label
    x_mean = coords[mask, 0].mean()
    y_mean = coords[mask, 1].mean()
    plt.text(
        x_mean, y_mean, str(label),
        fontsize=9, weight='bold',
        ha='center', va='center')

plt.show()

In [ ]:
# save the slide with all the annotated sections
adata_roi.write("/path/to/your/output/annotated_slide.h5ad")

In [ ]:
# Save each manually-annotated section as its own h5ad file
import scanpy as sc
import os

# Output directory
save_dir = "/path/to/your/output/per_section/"
os.makedirs(save_dir, exist_ok=True)

# Ensure manual_anno exists
if "manual_anno" not in adata_roi.obs.columns:
    raise KeyError("'manual_anno' column not found in adata_roi.obs")

# Split and save
for cluster in adata_roi.obs["manual_anno"].unique():
    cluster_str = str(cluster)
    adata_cluster = adata_roi[adata_roi.obs["manual_anno"] == cluster].copy()
    file_path = os.path.join(save_dir, f"{cluster_str}.h5ad")
    adata_cluster.write_h5ad(file_path)
    print(f"✅ Saved {cluster_str} ({adata_cluster.n_obs} cells) to {file_path}")